# Mastering Document Ingestion: Advanced PDF Loading for RAG Pipelines

In Retrieval-Augmented Generation (RAG), the quality of the retrieved context is paramount to the final answer's accuracy. While simple text files are straightforward, real-world knowledge bases often reside in complex formats like PDFs. These documents are not merely streams of text; they contain intricate layouts, embedded images, and structured data within tables. Simply dumping a PDF into an embedding model will result in fragmented, inaccurate context that cripples the RAG system's performance.

This notebook tackles this critical first step: robust document ingestion. We move beyond basic loaders to explore specialized tools like `PDFPlumber` and OCR-enhanced methods. By learning how to correctly initialize these advanced loaders—specifying modes, enabling image extraction, and selecting appropriate parsers—you learn how to transform opaque PDF files into clean, structured, and contextually rich document objects. This mastery is foundational for building reliable RAG pipelines.

For advanced development using LangGraph, the ability to reliably parse documents means you can build sophisticated agents that don't just retrieve text, but understand *structure*. You will learn techniques to handle multi-modal content (text alongside images) and extract specific data types (like tables), ensuring your retrieval step provides actionable context rather than just noise.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Differentiate Loaders:** Select the appropriate LangChain document loader (`PyPDFLoader`, `PDFMinerLoader`, `PDFPlumberLoader`) based on the specific structural needs of a PDF file (e.g., tables vs. pure text).
*   **Handle Complex Content:** Implement advanced loading techniques to extract non-textual data, including images and structured table content, using OCR parsers.
*   **Inspect Metadata:** Effectively inspect document metadata (page number, source, etc.) to filter, chunk, or contextualize retrieved documents within a RAG workflow.
*   **Optimize Ingestion:** Understand the trade-offs between different loading modes (`mode='page'` vs. full document load) and how these choices impact context quality for downstream embedding models.


In [ ]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from pathlib import Path
from pprint import pp

### File Path Setup and Validation

This cell sets up the file path for the PDF document using `pathlib.Path` and then uses `.exists()` to verify that the specified knowledge source file is present in the expected directory structure before proceeding with loading.


In [24]:
# create the path for pdf file

# Initialize a Path object pointing to the relative location of the PDF file.
file_path = Path("../knowledge-source/attention_is_all_you_need.pdf")

# Check if the file exists at the specified path. This is crucial for preventing runtime errors later in the notebook.
file_path.exists()


True

## PyPDF Loader

### Code Explanation

This cell converts a `Path` object (likely from the `pathlib` module) into a standard, platform-agnostic string representation using POSIX format (`/`). This is often necessary when passing file paths to external libraries or APIs that expect standard strings rather than specialized path objects.


In [25]:
file_path.as_posix()


'../knowledge-source/attention_is_all_you_need.pdf'

### PDF Loading

This cell initializes a `PyPDFLoader` object. This loader is responsible for reading and parsing content from a local PDF file (`file_path`), specifically setting the mode to `'page'` to ensure that each page of the document is treated as a separate unit during loading.


In [26]:
# create a loader

pypdf_loader = PyPDFLoader(file_path=file_path.as_posix(), # Specify the path to the PDF file using its POSIX representation
                           mode='page') # Set the mode to 'page' so that each page is loaded as an individual document chunk


### Document Loading

This cell initializes the process by loading documents from a specified source using `pypdf_loader.load()`. This function is crucial because it transforms raw file data (like PDFs) into structured, usable document objects that can be processed by the RAG pipeline.


In [27]:
# load the documents

documents = pypdf_loader.load()


### Code Explanation

This cell simply prints the length of the `documents` variable, which is assumed to be a list or collection of loaded documents. It's used for quick verification to ensure that the document loading process was successful and that the expected number of sources are available for subsequent RAG steps.


In [28]:
# number of documents

len(documents)  # Calculates and prints the total count of documents in the 'documents' variable.


15

### Metadata Inspection

This cell prints the metadata associated with the first document (`documents[0]`) loaded into memory. Inspecting metadata is crucial in RAG pipelines as it often contains source information (e.g., file name, page number) necessary for citation and debugging.


In [29]:
pp(documents[0].metadata)


{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live '
                    '2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '../knowledge-source/attention_is_all_you_need.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}


### Code Explanation

This cell accesses and prints the raw text content of the second document (index 1) loaded into the `documents` list. It is necessary to inspect the structure and content of the loaded documents, ensuring that the PDF loader successfully extracted the desired page data.


In [30]:
# fetch the first page text content

# Accesses the element at index 1 (the second document/page) in the 'documents' list.
# The '.page_content' attribute is used to extract the raw string text from that specific Document object.
print(documents[0].page_content)


Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Exp

### Metadata Inspection

This cell uses the `pp` function (likely a custom helper or utility for pretty printing) to display the metadata associated with the fifth document (`documents[4]`). Inspecting this metadata is crucial in RAG pipelines to understand how source information (like page numbers, file names, or extraction dates) was captured, ensuring the retrieved context is properly attributed.


In [31]:
# metadata

# Use 'pp' (pretty print function) to display the metadata dictionary.
# This inspects the metadata of the document at index 4 in the 'documents' list.
pp(documents[4].metadata)


{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live '
                    '2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '../knowledge-source/attention_is_all_you_need.pdf',
 'total_pages': 15,
 'page': 4,
 'page_label': '5'}


In [38]:
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

### Image-Aware PDF Loading

This cell initializes a specialized `PyPDFLoader` instance. By setting `extract_images=True`, we instruct the loader to not only extract text but also images from the PDF, which is crucial for advanced RAG systems that need multimodal context. The use of `RapidOCRBlobParser()` ensures robust image extraction and parsing.


In [39]:
# create pypdf instance which can extract images

pypdf_image_loader = PyPDFLoader(file_path=file_path.as_posix(),
                                 mode="page",
                                 extract_images=True,
                                 images_parser=RapidOCRBlobParser(),
                                 images_inner_format="html-img")


### Loading Documents with Images

This cell initializes the document loading process, specifically designed to handle PDFs that contain images. It uses `pypdf_image_loader` to load all necessary documents and associated image data into a variable called `documents_with_images`.


In [52]:
# load documents with images

documents_with_images = pypdf_image_loader.load()


[INFO] 2026-06-12 11:23:14,553 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-12 11:23:14,557 [RapidOCR] download_file.py:60: File exists and is valid: /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-12 11:23:14,558 [RapidOCR] main.py:65: Using /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-12 11:23:14,581 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-12 11:23:14,583 [RapidOCR] download_file.py:60: File exists and is valid: /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-12 11:23:14,583 [RapidOCR] main.py:65: Using /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/mo

### Code Explanation

This cell calculates and displays the total number of document objects (or chunks) that have been successfully loaded into the `documents_with_images` list. This count is crucial for verifying that all expected documents were processed and are ready for subsequent RAG steps.


In [53]:
# length of the document

len(documents_with_images)


15

### Code Explanation

This line accesses a specific document object, `page_with_image`, from the list or collection `documents_with_images`. It is necessary here to isolate and work with the third element (index 2) of the loaded documents, which presumably contains both text and associated image data for advanced RAG processing.


In [67]:
page_with_image = documents_with_images[2]
print(page_with_image.page_content)


Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512.
Decoder: The decoder is also composed of a stack of N = 6identical layers. 

In [68]:
page_with_image = documents_with_images[5]
print(page_with_image.page_content)

Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types. n is the sequence length, d is the representation dimension, k is the kernel
size of convolutions and r the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2 · d) O(1) O(1)
Recurrent O(n · d2) O(n) O(n)
Convolutional O(k · n · d2) O(1) O(logk(n))
Self-Attention (restricted) O(r · n · d) O(1) O(n/r)
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. Ther

## PDF Miner

In [69]:
from langchain_community.document_loaders import PDFMinerLoader

### PDF Document Loading (PDFMinerLoader)

This cell initializes the `PDFMinerLoader`, which is a specialized loader designed to extract content from PDF files. It uses advanced settings like `RapidOCRBlobParser` for image extraction and specifies an HTML format (`html-img`) to ensure rich, structured data capture.


In [70]:
# create the miner loader

pdfminer_loader = PDFMinerLoader(file_path=file_path.as_posix(), # Specify the path to the PDF file.
                                 mode="page",             # Process the document page by page.
                                 extract_images=True,      # Ensure that images within the PDF are extracted.
                                 images_parser=RapidOCRBlobParser(), # Use a robust parser (like RapidOCR) for image content extraction.
                                 images_inner_format="html-img") # Define how extracted images should be formatted in the output.


In [49]:
# load the documents

documents_with_images = pdfminer_loader.load()

[INFO] 2026-06-12 11:21:49,598 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-12 11:21:50,273 [RapidOCR] download_file.py:60: File exists and is valid: /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-12 11:21:50,275 [RapidOCR] main.py:65: Using /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-06-12 11:21:50,302 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-06-12 11:21:50,304 [RapidOCR] download_file.py:60: File exists and is valid: /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-06-12 11:21:50,304 [RapidOCR] main.py:65: Using /Users/kevin/Desktop/engineer/ai-engineer/lang-graph/course/.venv/lib/python3.11/site-packages/rapidocr/mo

This cell accesses and prints a truncated portion of the `page_content` from a specific document object (`documents_with_images[2]`). Since PDF loaders often embed image descriptions or OCR text within the page content, slicing the end (`[-450:]`) allows inspection of the most recent textual data, which is crucial for debugging how multimodal information (like captions or extracted text near images) was processed.


In [51]:
# image text in pdf

# Accesses the 'page_content' attribute of the third document object (index 2).
# The content is a list-like structure, and we are slicing the last 450 characters/elements.
print(documents_with_images[2].page_content[-450:])


 output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum

3

<img alt="Output
Probabilities
Softmax
Linear
Add &amp; Norm
Feed
Forward
Add &amp; Norm
Add &amp; Norm
Multi-Head
Feed
Attention
Forward
Nx
Add &amp; Norm
Nx
Add &amp; Norm
Masked
Multi-Head
Multi-Head
Attention
Attention
Positional
Positional
Encoding
Encoding
Input
Output
Embedding
Embedding
Inputs
Outputs
(shifted right) src="#" />


### Code Explanation

This cell accesses and prints the `page_content` attribute of a specific document object (index 5) within the `documents_with_images` list. This is crucial for inspecting the raw text extracted from a particular page, allowing verification of the loader's output before further processing.


In [57]:
# text from table

# Accesses the document at index 5 in the documents_with_images list.
# Then, it prints the 'page_content' attribute, which holds the extracted text for that page.
print(documents_with_images[5].page_content)


Table 1: Maximum path lengths, per-layer complexity and minimum number of sequential operations
for different layer types. n is the sequence length, d is the representation dimension, k is the kernel
size of convolutions and r the size of the neighborhood in restricted self-attention.
Layer Type Complexity per Layer Sequential Maximum Path Length
Operations
Self-Attention O(n2 · d) O(1) O(1)
Recurrent O(n · d2) O(n) O(n)
Convolutional O(k · n · d2) O(1) O(logk(n))
Self-Attention (restricted) O(r · n · d) O(1) O(n/r)
3.5 Positional Encoding
Since our model contains no recurrence and no convolution, in order for the model to make use of the
order of the sequence, we must inject some information about the relative or absolute position of the
tokens in the sequence. To this end, we add "positional encodings" to the input embeddings at the
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. Ther

### Metadata Inspection

This cell inspects the metadata of the first document loaded into `documents_with_images`. The `.metadata` attribute is a dictionary containing source information (like file path, page number, etc.), which is crucial for grounding and debugging RAG pipelines.


In [71]:
pp(documents_with_images[0].metadata)


{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX with hyperref',
 'creationdate': '2024-04-10T21:11:43+00:00',
 'author': '',
 'keywords': '',
 'moddate': '2024-04-10T21:11:43+00:00',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live '
                    '2023) kpathsea version 6.3.5',
 'subject': '',
 'title': '',
 'trapped': '/False',
 'source': '../knowledge-source/attention_is_all_you_need.pdf',
 'total_pages': 15,
 'page': 0,
 'page_label': '1'}


### Code Explanation

This loop iterates through a list of loaded documents (`documents_with_images`) and uses conditional logic to find the first document whose metadata indicates it originated from page 7. Once found, it prints the content of that specific document and immediately exits the loop using `break`.


In [72]:
for doc in documents_with_images:
    # Check if the current document's metadata specifies it came from page 7
    if doc.metadata["page"] == 7:
        # Print the actual text content of the document (the page content)
        print(doc.page_content)
        # Stop iterating immediately after finding and processing the target page
        break


Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU Training Cost (FLOPs)
EN-DE EN-FR EN-DE EN-FR
ByteNet [18] 23.75
Deep-Att + PosUnk [39] 39.2 1.0 · 1020
GNMT + RL [38] 24.6 39.92 2.3 · 1019 1.4 · 1020
ConvS2S [9] 25.16 40.46 9.6 · 1018 1.5 · 1020
MoE [32] 26.03 40.56 2.0 · 1019 1.2 · 1020
Deep-Att + PosUnk Ensemble [39] 40.4 8.0 · 1020
GNMT + RL Ensemble [38] 26.30 41.16 1.8 · 1020 1.1 · 1021
ConvS2S Ensemble [9] 26.36 41.29 7.7 · 1019 1.2 · 1021
Transformer (base model) 27.3 38.1 3.3 · 1018
Transformer (big) 28.4 41.8 2.3 · 1019
Residual Dropout We apply dropout [33] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we apply dropout to the sums of the embeddings and the
positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of
Pdrop = 0.1.


## PDF Plumber

In [73]:
from langchain_community.document_loaders import PDFPlumberLoader

### PDF Loading and Initialization

This cell initializes a specialized document loader, `PDFPlumberLoader`, which is designed to extract text content from PDF files using the `pdfplumber` library. It takes the file path (converted to a POSIX format) as input, preparing the document for subsequent chunking and embedding processes.


In [75]:
# create the loader

# Initialize the specialized loader class.
# We use PDFPlumberLoader because it efficiently extracts text from PDFs.
plumber_loader = PDFPlumberLoader(file_path=file_path.as_posix())


ImportError: pdfplumber package not found, please install it with `pip install pdfplumber`

### Loading Documents

This cell initializes the document corpus by calling the `load()` method on the `plumber_loader` object. This function is responsible for ingesting and preparing all source documents (e.g., PDFs, text files) into a structured format suitable for retrieval-augmented generation (RAG).


In [ ]:
# load the docs

# Call the load method on the plumber_loader instance to ingest and process all source documents.
documents_with_metadata = plumber_loader.load()


### Code Explanation

This cell simply calculates and displays the total number of documents stored in the `documents_with_metadata` list or variable. This is a crucial step for verifying that the document loading process was successful and that all expected data chunks were loaded into memory before proceeding with embedding or indexing.


In [ ]:
len(documents_with_metadata) # Calculates the length of the documents_with_metadata collection to confirm the total number of loaded documents.


15

In [ ]:
print(documents_with_metadata[1].page_content)

1 Introduction
Recurrentneuralnetworks,longshort-termmemory[13]andgatedrecurrent[7]neuralnetworks
inparticular,havebeenfirmlyestablishedasstateoftheartapproachesinsequencemodelingand
transductionproblemssuchaslanguagemodelingandmachinetranslation[35,2,5]. Numerous
effortshavesincecontinuedtopushtheboundariesofrecurrentlanguagemodelsandencoder-decoder
architectures[38,24,15].
Recurrentmodelstypicallyfactorcomputationalongthesymbolpositionsoftheinputandoutput
sequences. Aligningthepositionstostepsincomputationtime,theygenerateasequenceofhidden
statesh ,asafunctionoftheprevioushiddenstateh andtheinputforpositiont. Thisinherently
t t−1
sequentialnatureprecludesparallelizationwithintrainingexamples,whichbecomescriticalatlonger
sequencelengths,asmemoryconstraintslimitbatchingacrossexamples. Recentworkhasachieved
significantimprovementsincomputationalefficiencythroughfactorizationtricks[21]andconditional
computation[32],whilealsoimprovingmodelperformanceincaseofthelatter. Thefundamental
const

### Code Explanation

This cell accesses and prints the metadata dictionary of the first document in the `documents_with_metadata` list. Metadata often contains crucial context (like source file name, page number, or creation date) that is vital for advanced RAG systems to improve grounding and traceability.


In [ ]:
pp(documents_with_metadata[0].metadata)


{'source': '../knowledge-source/attention_is_all_you_need.pdf',
 'file_path': '../knowledge-source/attention_is_all_you_need.pdf',
 'page': 0,
 'total_pages': 15,
 'Author': '',
 'CreationDate': 'D:20240410211143Z',
 'Creator': 'LaTeX with hyperref',
 'Keywords': '',
 'ModDate': 'D:20240410211143Z',
 'PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live '
                    '2023) kpathsea version 6.3.5',
 'Producer': 'pdfTeX-1.40.25',
 'Subject': '',
 'Title': '',
 'Trapped': 'False'}
